In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前设备: {device}")

In [ ]:
from transformers import ChineseCLIPProcessor, ChineseCLIPModel

model_id = "AI-ModelScope/chinese-clip-vit-base-patch16"

print(f"正在加载模型: {model_id} ...")
model = ChineseCLIPModel.from_pretrained(model_id).to(device)
processor = ChineseCLIPProcessor.from_pretrained(model_id)

In [ ]:
from PIL import Image

image_path = "dog.jpg"

try:
    image = Image.open(image_path).convert("RGB")
except FileNotFoundError:
    print(f"错误：找不到文件 {image_path}，请确保图片存在。")
    exit()

In [ ]:
candidates = [
    "一张狗的照片",
    "一张猫的照片",
    "一张鱼的照片",
    "一张鸟的照片",
    "一张人的照片"
]

print(f"\n正在对图片进行分类...")

In [ ]:
inputs = processor(
    text=candidates,
    images=image,
    return_tensors="pt",
    padding=True
).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    # 获取图像与文本的相似度分数
    logits_per_image = outputs.logits_per_image
    # 将分数转换为概率 (Softmax)
    probs = logits_per_image.softmax(dim=1)


In [ ]:
print("\n--- 分类结果 ---")
for i, candidate in enumerate(candidates):
    print(f"{candidate}: {probs[0][i]:.4f}")

best_match_idx = probs.argmax()
print(f"\n 预测结果： **{candidates[best_match_idx]}** (置信度: {probs[0][best_match_idx]:.2%})")